In [1]:
#%pip uninstall attr attrs -y
#%pip install attrs fastf1

StatementMeta(, c0ba5334-d691-4545-b44e-631f96e198b9, 5, Finished, Available, Finished, False)

In [2]:
#import necessary packages
import fastf1
import pandas as pd
import os

StatementMeta(, c0ba5334-d691-4545-b44e-631f96e198b9, 6, Finished, Available, Finished, False)

In [3]:
# These are default values. The Data Pipeline will securely OVERWRITE these during an automated run!
p_year = '2025'
p_track = 'Bahrain'
p_session = 'FP2'

StatementMeta(, c0ba5334-d691-4545-b44e-631f96e198b9, 7, Finished, Available, Finished, False)

**Parameters and Setup**

In [4]:
# 1. Setup the Cache
cache_dir = '/lakehouse/default/Files/fastf1_cache'
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

# 2. Parse the pipeline parameter (converts "FP1,FP2,FP3,Q,R" into a Python list)
# If p_session doesn't exist (running manually), default to FP2
sessions_to_run = p_session.split(',') if 'p_session' in locals() else ['FP2']

clean_track = p_track.replace(" ", "_").lower()

# 3. THE WEEKEND LOOP: Process every session for this track
for current_session in sessions_to_run:
    print(f"\n--- Attempting to download data for: {p_year} {p_track} {current_session} ---")
    
    try:
        # Try to download the specific session
        session = fastf1.get_session(int(p_year), p_track, current_session)
        session.load(telemetry=False, weather=True)

        # Extract Laps and Weather
        laps_df = session.laps
        weather_df = session.weather_data

        # Convert to Strings to prevent translation errors
        laps_text_df = laps_df.astype(str)
        weather_text_df = weather_df.astype(str)

        # Convert to Spark
        spark_laps = spark.createDataFrame(laps_text_df)
        spark_weather = spark.createDataFrame(weather_text_df)

        # Create dynamic table names (e.g., raw_laps_2025_bahrain_q)
        laps_table_name = f"raw_laps_{p_year}_{clean_track}_{current_session.lower()}"
        weather_table_name = f"raw_weather_{p_year}_{clean_track}_{current_session.lower()}"

        # Save to Lakehouse
        spark_laps.write.format("delta").mode("overwrite").saveAsTable(laps_table_name)
        spark_weather.write.format("delta").mode("overwrite").saveAsTable(weather_table_name)

        print(f"✅ Success! Saved {laps_table_name} and {weather_table_name}.")

    except Exception as e:
        # If the session doesn't exist (e.g., trying to pull FP3 on a Sprint weekend)
        print(f"⚠️ SKIPPED: Could not load {current_session} for {p_track}.")
        print(f"Reason: {e}")

StatementMeta(, c0ba5334-d691-4545-b44e-631f96e198b9, 8, Finished, Available, Finished, False)

core           INFO 	Loading data for Bahrain Grand Prix - Practice 2 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 

Success! Saved Bahrain to the Lakehouse.
